# Notebook 07: Donation Forecasting (Predictive Regression)

## Section 1 — Problem Framing

**Business Problem:** Forecast future monthly donation totals to support financial planning and identify months that need extra fundraising attention.

**Approach:** Predictive regression using time-based features (month, lag values, rolling averages). We treat this as a regression problem rather than time-series ARIMA because the limited number of data points (~36 months) makes ARIMA unreliable.

**Target Variable:** Monthly total monetary donation amount (PHP)

**Success Metrics:** RMSE (lower is better), R-squared (higher is better)

**Stakeholders:** Financial planning staff, fundraising team

## Section 2 — Data Acquisition and Preparation

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
# Load donations data
donations = pd.read_csv('../../data/lighthouse_csv_v7/donations.csv')
print(f"Total donations: {len(donations)}")
print(f"Donation types: {donations['donation_type'].value_counts().to_dict()}")

# Filter to monetary donations only (have numeric amount values)
monetary = donations[donations['donation_type'] == 'Monetary'].copy()
monetary['donation_date'] = pd.to_datetime(monetary['donation_date'])
print(f"\nMonetary donations: {len(monetary)}")
print(f"Date range: {monetary['donation_date'].min()} to {monetary['donation_date'].max()}")
print(f"Amount stats:")
print(monetary['amount'].describe())

Total donations: 420
Donation types: {'Monetary': 234, 'InKind': 98, 'Time': 46, 'SocialMedia': 23, 'Skills': 19}

Monetary donations: 234
Date range: 2023-01-13 00:00:00 to 2026-03-01 00:00:00
Amount stats:
count     234.00000
mean     1028.73735
std       767.59608
min       250.00000
25%       482.50750
50%       819.63000
75%      1333.15500
max      6481.54000
Name: amount, dtype: float64


In [3]:
# Aggregate to monthly totals
monthly = monetary.groupby(pd.Grouper(key='donation_date', freq='MS')).agg(
    total_amount=('amount', 'sum'),
    donation_count=('donation_id', 'count')
).reset_index()

# Remove months with zero donations (if any gaps)
monthly = monthly[monthly['donation_count'] > 0].copy()
monthly = monthly.sort_values('donation_date').reset_index(drop=True)

print(f"Monthly aggregated records: {len(monthly)}")
print(f"Date range: {monthly['donation_date'].min()} to {monthly['donation_date'].max()}")
print(f"\nMonthly total_amount stats:")
print(monthly['total_amount'].describe())

Monthly aggregated records: 39
Date range: 2023-01-01 00:00:00 to 2026-03-01 00:00:00

Monthly total_amount stats:
count       39.000000
mean      6172.424103
std       4201.743504
min        342.960000
25%       3172.570000
50%       5309.040000
75%       7911.285000
max      17960.360000
Name: total_amount, dtype: float64


In [4]:
# Engineer time-based features
monthly['month'] = monthly['donation_date'].dt.month       # 1-12, for seasonality
monthly['year'] = monthly['donation_date'].dt.year
monthly['month_index'] = range(len(monthly))                # Sequential index for trend

# Lag features
monthly['lag_1'] = monthly['total_amount'].shift(1)         # Previous month total
monthly['lag_3'] = monthly['total_amount'].shift(3)         # 3-month lag

# Rolling averages
monthly['rolling_mean_3'] = monthly['total_amount'].rolling(window=3).mean()
monthly['rolling_mean_6'] = monthly['total_amount'].rolling(window=6).mean()

# Drop rows with NaN from lag/rolling features
monthly_clean = monthly.dropna().copy()
print(f"Records after dropping NaN (lag/rolling): {len(monthly_clean)}")
print(f"\nFeatures created:")
for col in ['month', 'year', 'month_index', 'lag_1', 'lag_3', 'rolling_mean_3', 'rolling_mean_6', 'donation_count']:
    print(f"  - {col}")
print(f"\nSample data:")
print(monthly_clean.head())

Records after dropping NaN (lag/rolling): 34

Features created:
  - month
  - year
  - month_index
  - lag_1
  - lag_3
  - rolling_mean_3
  - rolling_mean_6
  - donation_count

Sample data:
  donation_date  total_amount  donation_count  month  year  month_index  \
5    2023-06-01       1306.86               2      6  2023            5   
6    2023-07-01       6592.33               8      7  2023            6   
7    2023-08-01       4897.48               7      8  2023            7   
8    2023-09-01       2676.03               3      9  2023            8   
9    2023-10-01       4543.20               4     10  2023            9   

     lag_1    lag_3  rolling_mean_3  rolling_mean_6  
5  4862.09  9577.83     3856.806667     4098.886667  
6  1306.86  5401.47     4253.760000     4967.621667  
7  6592.33  4862.09     4265.556667     5439.676667  
8  4897.48  1306.86     4721.946667     4289.376667  
9  2676.03  6592.33     4038.903333     4146.331667  


In [5]:
# Define features and target
feature_cols = ['month', 'year', 'month_index', 'lag_1', 'lag_3', 
                'rolling_mean_3', 'rolling_mean_6', 'donation_count']

X = monthly_clean[feature_cols]
y = monthly_clean['total_amount']

# Temporal split — last 20% of months as test (DO NOT shuffle time series)
split_idx = int(len(monthly_clean) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training set: {len(X_train)} months ({monthly_clean['donation_date'].iloc[0].strftime('%Y-%m')} to {monthly_clean['donation_date'].iloc[split_idx-1].strftime('%Y-%m')})")
print(f"Test set: {len(X_test)} months ({monthly_clean['donation_date'].iloc[split_idx].strftime('%Y-%m')} to {monthly_clean['donation_date'].iloc[-1].strftime('%Y-%m')})")

Training set: 27 months (2023-06 to 2025-08)
Test set: 7 months (2025-09 to 2026-03)


## Section 3 — Exploration

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Line chart of monthly donation totals over time
axes[0, 0].plot(monthly['donation_date'], monthly['total_amount'], marker='o', color='coral', markersize=4)
axes[0, 0].set_title('Monthly Donation Totals Over Time')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Total Amount (PHP)')
axes[0, 0].tick_params(axis='x', rotation=45)

# Seasonal pattern (mean by month)
seasonal = monthly.groupby('month')['total_amount'].mean()
axes[0, 1].bar(seasonal.index, seasonal.values, color='teal', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Average Donations by Month (Seasonality)')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Mean Total Amount (PHP)')
axes[0, 1].set_xticks(range(1, 13))

# Rolling average overlay
axes[1, 0].plot(monthly['donation_date'], monthly['total_amount'], alpha=0.5, label='Actual', color='gray')
if 'rolling_mean_3' in monthly.columns:
    axes[1, 0].plot(monthly['donation_date'], monthly['rolling_mean_3'], label='3-month MA', color='coral', linewidth=2)
    axes[1, 0].plot(monthly['donation_date'], monthly['rolling_mean_6'], label='6-month MA', color='teal', linewidth=2)
axes[1, 0].set_title('Donations with Moving Averages')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Amount (PHP)')
axes[1, 0].legend()
axes[1, 0].tick_params(axis='x', rotation=45)

# Distribution of monthly totals
axes[1, 1].hist(monthly['total_amount'], bins=15, edgecolor='black', alpha=0.7, color='coral')
axes[1, 1].set_title('Distribution of Monthly Donation Totals')
axes[1, 1].set_xlabel('Total Amount (PHP)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../../ml-pipelines/notebooks/07_exploration.png', dpi=100, bbox_inches='tight')
plt.show()
print("Exploration plots generated")

Exploration plots generated


## Section 4 — Modeling (Regression Model Progression)

In [7]:
# Define preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), feature_cols)
])

# Model progression — compare multiple regression approaches
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree(depth=3)': DecisionTreeRegressor(max_depth=3, random_state=42),
    'RandomForest(n=100,depth=3)': RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42),
    'GradientBoosting(n=100,depth=2)': GradientBoostingRegressor(n_estimators=100, max_depth=2, random_state=42)
}

results = {}
cv_folds = min(5, len(X_train))  # Use min in case train set is very small

print("Model Comparison (Cross-Validation on Training Set):")
print("=" * 70)
for name, regressor in models.items():
    pipeline = SkPipeline([
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])
    
    # Cross-validate
    cv_scores = cross_val_score(pipeline, X_train, y_train, 
                                 cv=cv_folds, scoring='neg_mean_squared_error')
    cv_rmse = np.sqrt(-cv_scores)
    
    # Fit on full training set and evaluate on test
    pipeline.fit(X_train, y_train)
    y_pred_test = pipeline.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_r2 = r2_score(y_test, y_pred_test)
    
    results[name] = {
        'cv_rmse_mean': cv_rmse.mean(),
        'cv_rmse_std': cv_rmse.std(),
        'test_rmse': test_rmse,
        'test_r2': test_r2,
        'pipeline': pipeline
    }
    
    print(f"  {name}:")
    print(f"    CV RMSE: {cv_rmse.mean():.2f} (+/- {cv_rmse.std():.2f})")
    print(f"    Test RMSE: {test_rmse:.2f}")
    print(f"    Test R-squared: {test_r2:.4f}")
    print()

# Select best model by CV RMSE
best_name = min(results, key=lambda k: results[k]['cv_rmse_mean'])
best_pipeline = results[best_name]['pipeline']
print(f"BEST MODEL: {best_name}")
print(f"  CV RMSE: {results[best_name]['cv_rmse_mean']:.2f}")
print(f"  Test RMSE: {results[best_name]['test_rmse']:.2f}")
print(f"  Test R-squared: {results[best_name]['test_r2']:.4f}")

Model Comparison (Cross-Validation on Training Set):
  LinearRegression:
    CV RMSE: 2208.66 (+/- 651.88)
    Test RMSE: 2954.54
    Test R-squared: 0.4520

  DecisionTree(depth=3):
    CV RMSE: 2609.50 (+/- 1261.01)
    Test RMSE: 2950.49
    Test R-squared: 0.4535



  RandomForest(n=100,depth=3):
    CV RMSE: 2268.15 (+/- 480.49)
    Test RMSE: 2028.48
    Test R-squared: 0.7417

  GradientBoosting(n=100,depth=2):
    CV RMSE: 1869.69 (+/- 289.78)
    Test RMSE: 1929.77
    Test R-squared: 0.7662

BEST MODEL: GradientBoosting(n=100,depth=2)
  CV RMSE: 1869.69
  Test RMSE: 1929.77
  Test R-squared: 0.7662


## Section 5 — Evaluation

In [8]:
# Test set evaluation with best model
y_pred_best = best_pipeline.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_best))
test_r2 = r2_score(y_test, y_pred_best)

print(f"Best Model: {best_name}")
print(f"Test RMSE: {test_rmse:.2f} PHP")
print(f"Test R-squared: {test_r2:.4f}")
print(f"Mean monthly total: {y.mean():.2f} PHP")
print(f"RMSE as % of mean: {test_rmse/y.mean()*100:.1f}%")

Best Model: GradientBoosting(n=100,depth=2)
Test RMSE: 1929.77 PHP
Test R-squared: 0.7662
Mean monthly total: 6395.24 PHP
RMSE as % of mean: 30.2%


In [9]:
# Actual vs predicted line chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Actual vs Predicted
test_dates = monthly_clean['donation_date'].iloc[split_idx:].values
axes[0].plot(test_dates, y_test.values, marker='o', label='Actual', color='coral')
axes[0].plot(test_dates, y_pred_best, marker='s', label='Predicted', color='teal', linestyle='--')
axes[0].set_title(f'Actual vs Predicted Monthly Donations ({best_name})')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Total Amount (PHP)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Residual plot
residuals = y_test.values - y_pred_best
axes[1].scatter(y_pred_best, residuals, color='coral', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].set_title('Residuals vs Predicted')
axes[1].set_xlabel('Predicted Amount (PHP)')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.savefig('../../ml-pipelines/notebooks/07_evaluation.png', dpi=100, bbox_inches='tight')
plt.show()

## Section 6 — Feature Importance and Interpretation

In [10]:
# Feature importance from best model
regressor = best_pipeline.named_steps['regressor']

if hasattr(regressor, 'feature_importances_'):
    importances = regressor.feature_importances_
    feature_imp = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("Feature Importance (from best model):")
    print(feature_imp.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(feature_imp['Feature'], feature_imp['Importance'], color='coral', edgecolor='black', alpha=0.7)
    ax.set_title(f'Feature Importance ({best_name})')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('../../ml-pipelines/notebooks/07_importance.png', dpi=100, bbox_inches='tight')
    plt.show()
elif hasattr(regressor, 'coef_'):
    coefs = pd.DataFrame({
        'Feature': feature_cols,
        'Coefficient': regressor.coef_
    }).sort_values('Coefficient', ascending=False)
    
    print("Feature Coefficients (LinearRegression):")
    print(coefs.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(coefs['Feature'], coefs['Coefficient'], color='coral', edgecolor='black', alpha=0.7)
    ax.set_title(f'Feature Coefficients ({best_name})')
    ax.set_xlabel('Coefficient Value')
    plt.tight_layout()
    plt.savefig('../../ml-pipelines/notebooks/07_importance.png', dpi=100, bbox_inches='tight')
    plt.show()

Feature Importance (from best model):
       Feature  Importance
donation_count    0.767613
rolling_mean_6    0.095560
rolling_mean_3    0.045326
         month    0.034465
          year    0.026183
         lag_3    0.016382
   month_index    0.011416
         lag_1    0.003055


### Discussion

**Lag Features vs Calendar Features:**
- Lag features (lag_1, lag_3, rolling averages) capture recent donation momentum and are typically the strongest predictors
- Calendar features (month, year) capture seasonality (e.g., end-of-year giving spikes) and long-term trends
- The combination of both provides the model with both short-term context and seasonal patterns

**Limitations:**
- **Small dataset:** With only ~30 monthly observations, the model may overfit to recent patterns
- **Assumes stationarity:** The model assumes past donation patterns will continue, which may not hold if fundraising strategies or donor base changes significantly
- **External factors:** Economic conditions, major events, and new fundraising campaigns are not captured
- **Lag feature leakage:** At prediction time, lag features require knowing the actual previous month's donations

### Recommendations
- **Months needing extra fundraising:** Identify months with historically low donations from the seasonal chart and target additional campaigns
- **Trend monitoring:** Track whether the model's predictions diverge from reality, signaling a shift in donation patterns
- **Retrain periodically:** Update the model quarterly with new donation data

## Section 7 - Deployment

**Deployment Architecture:** Pre-computed predictions written to PostgreSQL.

This model is deployed as an offline batch pipeline. The production workflow is:

1. **ETL:** `jobs/etl_donation_forecast.py` reads donations from PostgreSQL, creates monthly aggregates with lag features, and writes the `ml_donation_forecast_features` table.
2. **Train:** `jobs/train_donation_forecast.py` trains a GradientBoostingRegressor pipeline, saves the model as `artifacts/donation_forecast.sav` along with `metadata.json` and `metrics.json`.
3. **Inference:** `jobs/run_inference_donation_forecast.py` loads the trained model, predicts monthly totals (including a next-month forecast), and writes results to the `donation_forecast_predictions` table in PostgreSQL.

The .NET backend queries `donation_forecast_predictions` via EF Core. The frontend fetches forecasts from `GET /api/predictions/donation-forecast`.

**Model type:** Predictive (regression)

**Approach:** Predictive -- forecast future donation amounts based on historical trends.

In [ ]:
# Save best pipeline as .sav to artifacts/
joblib.dump(best_pipeline, '../artifacts/donation_forecast.sav')
print("Model saved to: ../artifacts/donation_forecast.sav")

print("\nProduction scripts:")
print("  ETL:       jobs/etl_donation_forecast.py")
print("  Train:     jobs/train_donation_forecast.py")
print("  Inference: jobs/run_inference_donation_forecast.py")
print("\nPredictions are pre-computed to PostgreSQL table: donation_forecast_predictions")

In [12]:
# Expected input features for prediction
print("Expected input features (as DataFrame columns):")
print(f"  Features: {feature_cols}")
print()

# Example prediction
example = pd.DataFrame([{
    'month': 12,             # December
    'year': 2026,
    'month_index': len(monthly_clean),  # Next sequential month
    'lag_1': monthly_clean['total_amount'].iloc[-1],      # Last month's total
    'lag_3': monthly_clean['total_amount'].iloc[-3],      # 3 months ago
    'rolling_mean_3': monthly_clean['total_amount'].iloc[-3:].mean(),  # 3-month avg
    'rolling_mean_6': monthly_clean['total_amount'].iloc[-6:].mean(),  # 6-month avg
    'donation_count': monthly_clean['donation_count'].iloc[-1]  # Use last month's count as estimate
}])
pred = loaded.predict(example)
print(f"Example prediction for Dec 2026: {pred[0]:.2f} PHP")

Expected input features (as DataFrame columns):
  Features: ['month', 'year', 'month_index', 'lag_1', 'lag_3', 'rolling_mean_3', 'rolling_mean_6', 'donation_count']

Example prediction for Dec 2026: 3039.25 PHP
